In [ ]:
import signal
import wandb
import torch
import os 

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from hydra import compose, initialize

from codefiles.helpers import is_running_in_notebook  # for reloading modules instead of restarting kernel
if is_running_in_notebook():
    from codefiles import helpers
    import importlib
    importlib.reload(helpers)
from codefiles.helpers import set_all_seeds, signal_handler, build_model, build_lightningmodule, build_datamodule

os.environ["WANDB_SILENT"] = "true"
torch.set_float32_matmul_precision("high")

def main(cfg) -> None:
    wandb.finish()
    set_all_seeds(seed=cfg.seed)
    wandb.init(
        project=cfg.wandb.project,
        group=None if cfg.wandb.group == "None" else cfg.wandb.group,
        config={key: value for key, value in cfg.items()},
    )

    model = build_model(cfg)
    lightningmodule = build_lightningmodule(cfg, model)
    datamodule = build_datamodule(cfg)

    trainer = pl.Trainer(
        logger=WandbLogger(project=cfg.wandb.project, dir="wandb/"),
        log_every_n_steps=1,
        accelerator='gpu',
        devices=1,
        max_epochs=cfg.max_epochs,
        precision=cfg.precision
    )

    trainer.fit(lightningmodule, datamodule)
    wandb.finish()

if __name__ == "__main__":
    CONFIG_NAME = "config"
    signal.signal(signal.SIGINT, signal_handler)
    with initialize(version_base="1.1", config_path="config"):
        cfg = compose(config_name=f"{CONFIG_NAME}")
    main(cfg)

Seed set to 420
/sc-projects/sc-proj-ukb-cvd/environments/mml/lib/python3.9/site-packages/transformers/modeling_utils.py:1035: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  

total_samples: 1284 / 1284
no_missing: 899 / 1284
1_missing: 385 / 1284
modality_0_missing: 122 / 1284
modality_1_missing: 134 / 1284
modality_2_missing: 129 / 1284
total_samples: 229 / 229
no_missing: 161 / 229
1_missing: 68 / 229
modality_0_missing: 22 / 229
modality_1_missing: 23 / 229
modality_2_missing: 23 / 229
total_samples: 686 / 686
no_missing: 481 / 686
1_missing: 205 / 686
modality_0_missing: 64 / 686
modality_1_missing: 70 / 686
modality_2_missing: 71 / 686



   | Name        | Type                       | Params | Mode 
--------------------------------------------------------------------
0  | model       | Multimodal_Architecture    | 121 M  | train
1  | loss        | L1Loss                     | 0      | train
2  | acc_2_train | BinaryAccuracy             | 0      | train
3  | acc_2_val   | BinaryAccuracy             | 0      | train
4  | acc_2_test  | BinaryAccuracy             | 0      | train
5  | acc_7_train | MulticlassAccuracy         | 0      | train
6  | acc_7_val   | MulticlassAccuracy         | 0      | train
7  | acc_7_test  | MulticlassAccuracy         | 0      | train
8  | f1_train    | BinaryF1Score              | 0      | train
9  | f1_val      | BinaryF1Score              | 0      | train
10 | f1_test     | BinaryF1Score              | 0      | train
11 | ece         | MulticlassCalibrationError | 0      | train
12 | mce         | MulticlassCalibrationError | 0      | train
13 | rmsce       | MulticlassCalibrationError | 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [21]:
import torch 

x = [torch.randn(64, 1, 50) for _ in range(3)]

x[0][0, 0, :] = torch.nan
src_mask = [torch.isnan(x[i]).view(64, -1).any(1) for i in range(len(x))]
src_mask = torch.stack(src_mask, dim=-1)

transformer_input = torch.cat(x, dim=1)
transformer_input = torch.nan_to_num(transformer_input, nan=0.0)
transformer = torch.nn.TransformerEncoder(
    encoder_layer=torch.nn.TransformerEncoderLayer(d_model=50, nhead=1, batch_first=True),
    num_layers=1
)

out = transformer(transformer_input, src_key_padding_mask=src_mask)
print(out)

tensor([[[ 0.5878,  1.0370,  0.0577,  ...,  2.4688,  0.2920,  0.8688],
         [-0.8937, -1.8733, -0.8371,  ...,  0.5391,  1.0875,  0.8526],
         [-0.2392,  0.6350,  1.2745,  ...,  0.4297, -0.5186,  1.5135]],

        [[-0.5598, -0.9236, -0.5886,  ...,  0.7838,  0.1172, -1.3550],
         [-0.3545,  2.3954,  0.7412,  ...,  0.4345,  1.0314, -0.1962],
         [-0.8260,  1.2642,  0.7580,  ..., -0.7101, -0.5425, -1.4799]],

        [[-0.6412,  1.2532,  0.3484,  ..., -0.1112,  0.9876, -1.2037],
         [-0.1714,  1.8958, -0.6057,  ..., -0.5492,  1.6862, -1.4218],
         [-1.1986, -0.1063, -0.9114,  ..., -0.8649, -2.0944,  1.2927]],

        ...,

        [[-1.2250, -0.0759,  1.6379,  ...,  1.1694,  0.2616, -0.4867],
         [-0.9156, -0.7569,  0.3610,  ..., -1.5910,  0.4035,  0.0936],
         [ 0.7059,  0.3685,  2.9654,  ...,  0.6212,  2.0158, -0.7764]],

        [[ 1.4860, -1.0076, -0.9168,  ..., -0.8262, -0.7609, -0.7249],
         [-1.1037,  0.1358, -1.0061,  ...,  3.1428,  0.

/sc-projects/sc-proj-ukb-cvd/environments/mml/lib/python3.9/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(
